# BGE-M3 frozen + Ridge · tiga input · CV dan test

**Input:** soal, jawaban referensi, respons siswa. **Target:** satu skor guru `rater`0–100.
Eksperimen khusus BGE→Ridge: tidak menjalankan S1/S3 atau MLP. Ini lanjutan diagnosis readout, sekarang mencakup pemilihan alpha development, final refit dan evaluasi test.

Setup → upload Excel → audit → split → fitur frozen → nested CV → final fit → test → lima Excel dan ZIP.
Pilih **T4 GPU** untuk ekstraksi. Ridge dijalankan di CPU. Tanpa Google Drive, clone repo, atau file kode tambahan.
Jalankan dari atas dengan **Run all**, pilih `fix_dataset_300.xlsx`, lalu tunggu unduhan hasil.

**Belum ada klaim hasil penelitian dari notebook ini.** Angka baru tersedia setelah dijalankan pada dataset asli.

## 1 · Setup, ruang kerja dan upload

Worker numerik terpisah mempertahankan Torch CUDA Colab dan mencegah konflik NumPy kernel. Data/cache/model hanya berada di disk runtime lokal; unduh ZIP sebelum runtime berakhir.

In [ ]:
import os, sys, json, subprocess, html, secrets, atexit, argparse, hashlib, venv
from pathlib import Path
from IPython.display import display, HTML
try:
    from packaging.requirements import Requirement
except ImportError:
    from pip._vendor.packaging.requirements import Requirement
IN_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()
default_workspace='/content/AES_300_BGE_M3_Ridge_Tiga_Input_Colab' if IN_COLAB else str(Path.cwd()/'outputs/AES_300_BGE_M3_Ridge_Tiga_Input_Colab')
ROOT=Path(os.environ.get('AES300_WORKSPACE',default_workspace)).expanduser().absolute()
ROOT=ROOT.resolve()
if ROOT==Path('/content/drive') or Path('/content/drive') in ROOT.parents:
    raise ValueError('Workspace harus lokal; BGE Ridge tiga input tidak menulis Google Drive.')
ROOT.mkdir(parents=True,exist_ok=True)
def prepare_local_storage(workspace, in_colab):
    storage=Path(workspace).expanduser().resolve()
    if storage==Path('/content/drive') or Path('/content/drive') in storage.parents:
        raise ValueError('BGE Ridge tiga input hanya menyimpan lokal; pilih workspace di luar Google Drive.')
    storage.mkdir(parents=True,exist_ok=True)
    probe=storage/'.aes_write_check';probe.write_text('local write check');probe.unlink()
    return storage

# Identitas kode notebook untuk membedakan hasil dari implementasi yang berbeda.
NOTEBOOK_CODE_SHA256='548036d3faa9b8a0a9026abe02445b49e4589279e4c7baffebe67099631c9961'
print('Ruang kerja AES siap. Seluruh kode tersedia di notebook ini.')
print('Versi Python:',sys.version.split()[0])

# Hanya pustaka runtime yang diperlukan; alat pengujian tidak dipasang untuk menjalankan AES.
REQUIREMENTS_TEXT=r'''
# Torch dipertahankan dari runtime Colab supaya CUDA tidak diganti wheel CPU.
# Tanpa torch, setup memasang torch 2.6.0 CPU sebelum file ini.
numpy==2.2.6
pandas==2.2.3
scipy==1.15.3
scikit-learn==1.6.1
# tokenizers0.19 (transformers4.44) tidak menyediakan wheel Python3.13.
transformers==4.44.2; python_version == '3.12'
transformers==4.48.3; python_version == '3.13'
openpyxl==3.1.5
'''
(ROOT/'requirements.dataset300.txt').write_text(REQUIREMENTS_TEXT)

def resolve_dataset_upload(workspace, configured='', search_paths=None, browser_upload=False):
    """Web Colab: pakai file yang ada, atau tampilkan Choose Files pada cell ini."""
    from pathlib import Path
    import hashlib
    import os
    import shutil
    workspace=Path(workspace).expanduser().resolve()
    destination=workspace/'data/dataset300/fix_dataset_300.xlsx'
    if configured:
        candidates=[Path(configured).expanduser().resolve()]
        if not candidates[0].is_file():
            raise FileNotFoundError('DATASET_FILE/AES300_DATASET belum ditemukan: '+str(candidates[0]))
    else:
        places=search_paths if search_paths is not None else [Path('/content'),Path.cwd(),workspace]
        candidates=[]
        for place in places:
            path=(Path(place)/'fix_dataset_300.xlsx').resolve()
            if path.is_file() and path not in candidates and path!=destination:
                candidates.append(path)
        if not candidates and destination.is_file():
            candidates=[destination]
    if not candidates and browser_upload:
        from google.colab import files
        destination.parent.mkdir(parents=True,exist_ok=True)
        print('Klik Choose Files, lalu pilih fix_dataset_300.xlsx dari komputer.',flush=True)
        previous=Path.cwd()
        try:
            os.chdir(destination.parent)
            uploaded=files.upload()
        finally:
            os.chdir(previous)
        if 'fix_dataset_300.xlsx' not in uploaded:
            raise ValueError('Pilih fix_dataset_300.xlsx. Jika dibatalkan, jalankan ulang cell setup ini.')
        candidates=[destination]
        del uploaded
    if not candidates:
        raise FileNotFoundError(
            'Dataset belum ada. Di web Colab, jalankan cell setup ini dan pilih '
            'fix_dataset_300.xlsx melalui Choose Files. Alternatif: panel Files/Berkas kiri '
            '→ Upload, lalu jalankan ulang cell ini. Jika lokasinya berbeda, isi DATASET_FILE.')
    hashes={hashlib.sha256(p.read_bytes()).hexdigest() for p in candidates}
    if len(hashes)>1:
        raise ValueError('Ada beberapa dataset berbeda: '+', '.join(map(str,candidates))+
                         '. Isi DATASET_FILE dengan path file yang ingin dipakai.')
    source=candidates[0]
    if not source.stat().st_size:
        raise ValueError('File upload masih kosong; tunggu upload selesai lalu jalankan ulang cell ini.')
    destination.parent.mkdir(parents=True,exist_ok=True)
    if source!=destination:
        shutil.copy2(source,destination)
    print('Dataset dari upload lokal:',source,'→',destination,flush=True)
    return destination


In [ ]:
# Pemeriksaan dan pemasangan pustaka tersedia langsung di sini.
INSPECT = r'''
import json, sys
from importlib import metadata, util
try:
    from packaging.requirements import Requirement
except ImportError:
    from pip._vendor.packaging.requirements import Requirement
roots=json.loads(sys.argv[1])+['torch>=2.6.0']
versions={}
for raw in roots:
    name=Requirement(raw).name
    try:versions[name]=metadata.version(name)
    except metadata.PackageNotFoundError:versions[name]=None
queue=[Requirement(raw) for raw in roots];seen=set();issues=[];install_requirements=[]
while queue:
    req=queue.pop()
    if req.marker and not req.marker.evaluate({'extra':''}):continue
    try:dist=metadata.distribution(req.name)
    except metadata.PackageNotFoundError:
        issues.append(str(req)+' belum tersedia');install_requirements.append(str(req));continue
    if not req.specifier.contains(dist.version,prereleases=True):
        issues.append(str(req)+' diperlukan; tersedia '+dist.version);install_requirements.append(str(req))
    name=dist.metadata['Name'].lower().replace('_','-')
    if name in seen:continue
    seen.add(name)
    queue.extend(Requirement(raw) for raw in (dist.requires or []))
print(json.dumps({'python':sys.version.split()[0],'runtime_version':list(sys.version_info[:2]),
 'versions':versions,'torch_present':util.find_spec('torch') is not None,
 'dependency_issues':sorted(set(issues)),'install_requirements':sorted(set(install_requirements)),'dependency_count':len(seen)}))
'''

IMPORTS = r'''
def configure_text_only_transformers():
    """Hindari import vision ops yang tidak diperlukan oleh RoBERTa/BGE/LoRA."""
    import os
    os.environ['USE_TORCH'] = '1'
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    from transformers.utils import import_utils
    # Transformers 4.44/4.48 mendeteksi torchvision dari keberadaan paket,
    # meskipun binary ops-nya tidak cocok dengan torch runtime. Notebook ini
    # hanya teks: jangan muat backend gambar melalui import tidak langsung pustaka AES.
    # Hanya worker AES yang diubah; paket Colab dan operator torch tetap utuh.
    if not hasattr(import_utils, '_torchvision_available'):
        raise RuntimeError('Versi Transformers berbeda dari versi notebook yang didukung.')
    import_utils._torchvision_available = False
    return {'transformers_backend': 'pytorch_text_only', 'torchvision_enabled': False}

configure_text_only_transformers()
import json
import torch, numpy, pandas, scipy, sklearn, openpyxl
from transformers import AutoModel, AutoTokenizer
available=torch.cuda.is_available()
print(json.dumps({'torch':torch.__version__,'cuda_available':available,
 'device':torch.cuda.get_device_name(0) if available else 'CPU',
 'checkpoint_loaded':False,'imports_verified':True,'transformers_backend':'pytorch_text_only'}))
'''

def environment_folder(root, version):
    version=tuple(version[:2])
    if version not in ((3,12),(3,13)):
        raise RuntimeError('Setup AES mendukung Python 3.12 atau 3.13. Versi lain belum diuji.')
    return Path(root)/'outputs'/('dataset300_env_text_e6_gpu_reuse' if version==(3,12) else 'dataset300_env_py313_text_e6_gpu_reuse')


# Pilih versi pustaka sesuai Python aktif agar aturan Python 3.12 dan 3.13 tidak tertukar.
def active_requirements(path, version):
    """Hanya aturan untuk versi Python aktif; tidak mencoba semua versi Transformers."""
    selected=[]
    for line in Path(path).read_text().splitlines():
        line=line.split('#',1)[0].strip()
        if not line:continue
        req=Requirement(line)
        if req.marker and not req.marker.evaluate({'python_version':'.'.join(map(str,version[:2])), 'extra':''}):continue
        selected.append(req.name+str(req.specifier))
    return selected


class SetupError(RuntimeError):
    pass


# Memeriksa pustaka yang tersedia, menyiapkan versi yang diperlukan, lalu memvalidasi import sebelum melanjutkan.
def prepare_environment(argv=None):
    parser=argparse.ArgumentParser()
    parser.add_argument('--verify-python',help='Periksa worker yang disediakan pengguna tanpa memasang paket.')
    args=parser.parse_args(argv)
    folder=environment_folder(ROOT,sys.version_info)
    outputs=ROOT/'outputs';outputs.mkdir(parents=True,exist_ok=True)
    ready=outputs/'dataset300_setup.json';ready.unlink(missing_ok=True)
    logfile=outputs/'dataset300_setup.log'
    requirements=active_requirements(ROOT/'requirements.dataset300.txt',sys.version_info)
    env={**os.environ,'HF_HOME':str(outputs/'cache/huggingface'),'PIP_DISABLE_PIP_VERSION_CHECK':'1'}
    with logfile.open('w') as log:
        def execute(command,stage):
            log.write(stage+'\n'+json.dumps(command)+'\n');log.flush()
            process=subprocess.run(command,cwd=ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
            log.write(process.stdout or '');log.write(process.stderr or '');log.flush()
            if process.returncode:
                details='\n'.join((process.stderr or process.stdout or 'Proses gagal tanpa rincian.').splitlines()[-18:])
                raise SetupError(stage+' gagal.\n'+details+'\nRincian lengkap: '+str(logfile))
            return process.stdout
        def inspect(python):
            result=json.loads(execute([python,'-c',INSPECT,json.dumps(requirements)],'Pemeriksaan versi dan dependensi'))
            if result['runtime_version']!=list(sys.version_info[:2]):
                raise SetupError('Versi Python worker berbeda dari runtime. Gunakan lingkungan untuk versi runtime ini.')
            return result

        print('1/3 · Memeriksa lingkungan dan versi pustaka AES...',flush=True)
        if args.verify_python:
            # abspath menjaga symlink bin/python venv; resolve akan memakai Python global.
            python=str(Path(args.verify_python).absolute())
        else:
            if folder.exists() and not (folder/'pyvenv.cfg').is_file():
                raise SetupError('Folder target bukan lingkungan Python AES; tidak ditimpa.')
            if not (folder/'bin/python').exists():
                # Torch CUDA tetap berasal dari runtime. Paket AES yang berbeda dipasang lokal.
                venv.EnvBuilder(with_pip=False,system_site_packages=True).create(folder)
            python=str(folder/'bin/python')
        before=inspect(python)
        torch_version=before['versions'].get('torch')
        if torch_version and not Requirement('torch>=2.6.0').specifier.contains(torch_version,prereleases=True):
            raise SetupError('PyTorch minimal 2.6.0 diperlukan. Pilih runtime Colab yang lebih baru; torch bawaan tidak diganti.')
        if not before['torch_present']:
            raise SetupError('PyTorch bawaan runtime belum tersedia. Pilih runtime Colab yang baru; setup ini tidak memasang wheel CPU.')
        installed=False
        after=before
        if after['dependency_issues']:
            if args.verify_python:
                raise SetupError('Worker yang dipilih belum memenuhi kebutuhan AES. Rincian: '+str(logfile))
            print('2/3 · Menyiapkan pustaka AES tanpa mengganti PyTorch runtime...',flush=True)
            # --no-deps mencegah pustaka AES menarik ulang torch. Dependensi lain
            # diperiksa ulang dari metadata dan dipasang hanya jika diperlukan.
            pip=[sys.executable,'-m','pip','--python',python,'install','--ignore-installed','--no-deps','--no-input']
            for attempt in range(8):
                if not after['dependency_issues']:
                    break
                pending=after['install_requirements']
                if not pending or any(Requirement(raw).name.lower().replace('_','-')=='torch' for raw in pending):
                    raise SetupError('Dependensi meminta perubahan PyTorch. Gunakan runtime Colab yang kompatibel; torch bawaan tidak ditimpa.')
                execute(pip+pending,'Pemasangan pustaka AES privat, putaran '+str(attempt+1))
                installed=True
                after=inspect(python)
                if after['versions'].get('torch')!=torch_version:
                    raise SetupError('Versi PyTorch berubah selama setup; training tidak dimulai.')
        else:
            print('2/3 · Versi pustaka sudah cocok; tidak perlu instalasi ulang.',flush=True)
        if after['dependency_issues']:
            raise SetupError('Dependensi AES belum cocok: '+'; '.join(after['dependency_issues'][:3])+'. Rincian: '+str(logfile))
        # Jangan mengganti torch yang terpasang tetapi rusak dengan wheel CPU secara diam-diam.
        print('3/3 · Memeriksa import dan perangkat perhitungan...',flush=True)
        checked=json.loads(execute([python,'-c',IMPORTS],'Pemeriksaan import AES dan PyTorch'))
        result={**after,**checked,'python_version':after['python'],'python':python,
                'packages_action':'installed' if installed else 'reused','torch_reused':before['torch_present'],
                'log_file':str(logfile),'requirements_sha256':hashlib.sha256((ROOT/'requirements.dataset300.txt').read_bytes()).hexdigest()}
        ready.write_text(json.dumps(result,indent=2)+'\n')
    print('Persiapan AES selesai. Versi, dependensi, dan import sudah diperiksa.',flush=True)
    return result


In [ ]:
# Penghubung proses Python: menjaga variabel antar-cell %%aes tanpa file pendukung proyek.
WORKER_SOURCE=r'''
import json, sys, traceback, types
from pathlib import Path

prefix, workspace = sys.argv[1:3]
module = types.ModuleType('__aes_notebook__')
sys.modules[module.__name__] = module
namespace = module.__dict__
namespace['ROOT'] = Path(workspace)

def event(kind, **values):
    print(prefix+json.dumps({'kind':kind, **values},ensure_ascii=False,allow_nan=False),flush=True)

def show(frame, title='', rows=8, columns=8):
    # Fungsi tampilan diisi oleh cell notebook; tidak diimpor dari file proyek.
    import pandas as pd
    frame = frame if isinstance(frame,pd.DataFrame) else pd.DataFrame(frame)
    view = namespace['preview_table'](frame.columns,frame.head(rows).to_dict('records'),title,
                                    row_count=len(frame),max_rows=rows,max_columns=columns)
    view['rows'] = json.loads(pd.DataFrame(view['rows']).to_json(orient='values',force_ascii=False,default_handler=str))
    event('table',**view,html=namespace['render_table'](view))

namespace['show'] = show
event('ready',python=sys.version.split()[0])
for line in sys.stdin:
    try:
        payload = json.loads(line)
        if payload.get('command') == 'close':break
        exec(compile(payload['code'],payload.get('filename','<aes-cell>'),'exec'),namespace)
    except BaseException:
        event('error',traceback=traceback.format_exc())
    else:
        event('done')
'''


In [ ]:
DATASET_FILE=''  # Biasanya otomatis dari /content; isi path lengkap jika berbeda.
PYTHON = os.environ.get('AES300_PYTHON','')
def show_installation_log():
    path=ROOT/'outputs/dataset300_setup.log'
    contents=path.read_text() if path.is_file() else 'Log belum tersedia.'
    display(HTML('<details style="background:#f8fafc;color:#172035;padding:12px;border:1px solid #94a3b8">'
                 '<summary>Rincian instalasi — buka jika diperlukan</summary>'
                 '<pre style="max-height:320px;overflow:auto;white-space:pre-wrap;color:#172035">'
                 +html.escape(contents)+'</pre></details>'))
try:
    ready=prepare_environment(['--verify-python',PYTHON] if PYTHON else [])
except Exception:
    show_installation_log()
    raise
if ready['runtime_version'] != list(sys.version_info[:2]) or not ready['imports_verified']:
    raise RuntimeError('Metadata setup tidak cocok runtime aktif.')
PYTHON=ready['python']
setup_rows=[
    ('Status','Siap — versi, dependensi, dan import AES lulus pemeriksaan'),
    ('Python',ready['python_version']),
    ('PyTorch',ready['torch']),
    ('Perangkat','BGE: GPU jika tersedia; Ridge: CPU float64'),
    ('Pustaka AES','Versi yang cocok dipakai ulang' if ready['packages_action']=='reused'
                    else 'Versi khusus AES dipasang; pustaka bawaan tetap disimpan'),
    ('Pustaka utama','NumPy '+ready['versions']['numpy']+'; Transformers '+ready['versions']['transformers']),
    ('Model','Belum dimuat; tahap ini hanya menyiapkan lingkungan')]
status_html=''.join('<tr><th style="padding:9px;text-align:left;border:1px solid #94a3b8">'+html.escape(label)
                    +'</th><td style="padding:9px;border:1px solid #94a3b8">'+html.escape(value)+'</td></tr>'
                    for label,value in setup_rows)
display(HTML('<table style="background:#f8fafc!important;color:#172035!important;border-collapse:collapse">'
             +status_html+'</table>'))
show_installation_log()
PERSISTENT_ROOT=prepare_local_storage(ROOT,IN_COLAB)
print('Penyimpanan data, checkpoint dan Excel:',PERSISTENT_ROOT)
print('Checkpoint hanya lokal. Bisa dilanjutkan selama folder runtime masih ada; runtime dihapus berarti file lokal hilang. Unduh hasil sebelum menutup sesi.')
# Web Colab biasa: Choose Files muncul jika dataset belum diunggah.
DATA_PATH=resolve_dataset_upload(PERSISTENT_ROOT,DATASET_FILE or os.environ.get('AES300_DATASET',''),browser_upload=IN_COLAB)
def display_aes_table(event):
    display(HTML(event['html']))
if '_aes_process' in globals() and _aes_process.poll() is None:
    _aes_process.terminate()
_aes_prefix='__AES_'+secrets.token_hex(12)+'__'
child_env={**os.environ,'HF_HOME':str(ROOT/'outputs/cache/huggingface'),
           'AES300_DATASET':str(DATA_PATH),'AES300_RESULTS':str(PERSISTENT_ROOT/'outputs/dataset300_inline'),
           'AES300_REQUIRE_GPU':'0',
           'TOKENIZERS_PARALLELISM':'false',
           'OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2','PYTHONUNBUFFERED':'1'}
child_env.pop('PYTHONPATH',None)
_aes_process=subprocess.Popen([PYTHON,'-u','-c',WORKER_SOURCE,_aes_prefix,str(ROOT)],
                              cwd=ROOT,env=child_env,stdin=subprocess.PIPE,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1)
while True:
    line=_aes_process.stdout.readline()
    if not line:
        raise RuntimeError('Worker gagal mulai; lihat log setup.')
    if line.startswith(_aes_prefix):
        print('Proses perhitungan siap. Python',json.loads(line[len(_aes_prefix):])['python']);break
    print(line,end='')
def aes_cell(line,cell):
    if _aes_process.poll() is not None:
        raise RuntimeError('Worker berhenti; jalankan setup dan cell berikutnya dari awal.')
    _aes_process.stdin.write(json.dumps({'code':cell,'filename':'<notebook-cell>'})+'\n')
    _aes_process.stdin.flush()
    while True:
        output=_aes_process.stdout.readline()
        if not output:
            raise RuntimeError('Worker terhenti; jika RAM penuh pilih runtime lebih besar.')
        if not output.startswith(_aes_prefix):
            print(output,end='',flush=True);continue
        event=json.loads(output[len(_aes_prefix):])
        if event['kind']=='done':break
        if event['kind']=='error':raise RuntimeError(event['traceback'])
        if event['kind']=='table':display_aes_table(event)
get_ipython().register_magic_function(aes_cell,'cell','aes')
atexit.register(lambda:_aes_process.terminate() if _aes_process.poll() is None else None)
print('Dataset siap. Jalankan cell %%aes berurutan.')


## 2 · Parameter sebelum melihat hasil

Resep BGE mengikuti single-sequence E21: tiga teks, budget konteks192token (soal maksimal64), window1024 dengan overlap128, mean CLS antar-window lalu L2. Revision model dikunci.
Tidak memakai StandardScaler/target-centering. Ridge mempunyai intercept yang dipelajari hanya dari train dan alpha penalti L2. Tidak memiliki learning rate atau epoch.

Grid alpha adalah kandidat eksperimen yang diumumkan sebelum run, **bukan angka optimal dari paper**. Pemilihan meminimalkan pooled RMSE development; hasil ±1/±2/±5 tetap dilaporkan terpisah.

In [ ]:
%%aes
import os, sys, json, math, hashlib, random, warnings, re, itertools, unicodedata, zipfile, subprocess, gc, logging, html
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.functional import normalize
from torch.utils.data import DataLoader
def configure_text_only_transformers():
    """Hindari import vision ops yang tidak diperlukan oleh RoBERTa/BGE/LoRA."""
    import os
    os.environ['USE_TORCH'] = '1'
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    from transformers.utils import import_utils
    # Transformers 4.44/4.48 mendeteksi torchvision dari keberadaan paket,
    # meskipun binary ops-nya tidak cocok dengan torch runtime. Notebook ini
    # hanya teks: jangan muat backend gambar melalui import tidak langsung PEFT.
    # Hanya worker AES yang diubah; paket Colab dan operator torch tetap utuh.
    if not hasattr(import_utils, '_torchvision_available'):
        raise RuntimeError('Versi Transformers berbeda dari versi notebook yang didukung.')
    import_utils._torchvision_available = False
    return {'transformers_backend': 'pytorch_text_only', 'torchvision_enabled': False}

configure_text_only_transformers()
from transformers import AutoModel, AutoTokenizer
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from openpyxl import Workbook, load_workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

NOTEBOOK_CODE_SHA256='548036d3faa9b8a0a9026abe02445b49e4589279e4c7baffebe67099631c9961'
SCOPE='bge_m3_ridge_three_input_cv_and_test'
REQUIRED=['Email address','response_number','response_text','soal','jawaban_referensi','rater']

In [ ]:
%%aes
# Tentukan sebelum run; jangan ubah berdasarkan skor test historis.
FOLDS = 5
RIDGE_ALPHAS = [.01, .1, 1., 10., 100.]
BGE_DEVICE = 'auto'
EXPECTED_ROWS = 300
cfg = dict(dataset=os.environ['AES300_DATASET'], output=str(ROOT/'outputs/bge_ridge_three_input'),
    sheet='Sheet1', expected_rows=EXPECTED_ROWS, score_min=0, score_max=100,
    test_fraction=.2, folds=FOLDS, split_seed=2026, split_candidates=128,
    ridge_alphas=RIDGE_ALPHAS, bge_max_length=1024, bge_overlap=128, bge_batch=4,
    device=BGE_DEVICE, context_max_tokens=192, question_max_tokens=64, input_format='single_sequence',
    bge=dict(checkpoint='BAAI/bge-m3', revision='5617a9f61b028005a4858fdac845db406aefb181'))
if FOLDS < 3:
    raise ValueError('Nested CV memerlukan sedikitnya3fold.')
torch.set_num_threads(2)
print('Tiga input: soal + referensi + respons | BGE frozen | Ridge float64 CPU', flush=True)
print('Alpha:', RIDGE_ALPHAS, '| Seleksi: development saja', flush=True)

## 3 · Dataset asli dan satu label

Header wajib: `Email address`, `response_number`, `response_text`, `soal`, `jawaban_referensi`, `rater`.
Email dan duplikat membentuk kelompok pencegah kebocoran, tidak menjadi fitur. Label harus lengkap dan integer0–100; data tidak dipotong atau diisi otomatis.

In [ ]:
%%aes
def clean(text, normalize_punctuation=False):
    text = unicodedata.normalize('NFC', text).replace('\r\n','\n').replace('\r','\n')
    text = ''.join(c for c in text if c in '\n\t' or unicodedata.category(c) != 'Cc')
    if normalize_punctuation:
        text = text.translate(str.maketrans({'“':'"','”':'"','‘':"'",'’':"'",'–':'-','—':'-'})).replace('…','...')
    return text


def digest(path):
    # Checkpoint besar diperiksa bertahap supaya tidak dimuat seluruhnya ke RAM.
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            checksum.update(chunk)
    return checksum.hexdigest()


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.pending')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+'\n', encoding='utf-8')
    temporary.replace(path)


@dataclass
class Bundle:
    rows: list
    y: np.ndarray
    groups: np.ndarray
    source_sha256: str
    summary: dict

    @property
    def texts(self):
        return [r['text'] for r in self.rows]


def read_dataset(cfg):
    path = Path(cfg['dataset'])
    source_hash = digest(path)
    book = load_workbook(path, read_only=True, data_only=True)
    try:
        if cfg['sheet'] not in book.sheetnames:
            raise ValueError(f"Tab {cfg['sheet']} tidak tersedia: {book.sheetnames}")
        data = iter(book[cfg['sheet']].iter_rows(values_only=True))
        header = [str(x).strip() if x is not None else None for x in next(data)]
        if any(header.count(x) != 1 for x in REQUIRED):
            raise ValueError('Enam header wajib hilang atau duplikat: '+', '.join(REQUIRED))
        rows = []
        for excel_row, values in enumerate(data, 2):
            if all(v is None for v in values):
                continue  # Hanya baris format kosong, bukan label/jawaban yang hilang.
            raw = {k: values[i] for i, k in enumerate(header) if k is not None}
            if any(raw[k] is None or not str(raw[k]).strip() for k in REQUIRED):
                raise ValueError(f'Baris {excel_row}: kolom wajib kosong: '+', '.join(k for k in REQUIRED if raw[k] is None)+'; tidak diisi otomatis.')
            score = raw['rater']
            if isinstance(score, bool) or not isinstance(score, (int, float)) or not math.isfinite(score):
                raise ValueError(f'Baris {excel_row}: rater harus angka skor guru, bukan teks/ID.')
            if int(score) != score or not cfg['score_min'] <= score <= cfg['score_max']:
                raise ValueError(f'Baris {excel_row}: skor bukan kategori integer dalam rentang yang dinyatakan.')
            for key in ('soal','jawaban_referensi'):
                if not isinstance(raw[key],str) or not raw[key].strip():
                    raise ValueError(f'Baris {excel_row}: {key} harus teks yang tidak kosong.')
            if not isinstance(raw['response_text'], str):
                raise ValueError(f'Baris {excel_row}: jawaban bukan teks.')
            rows.append({'row_id': f'row_{excel_row-1:04d}', 'source_row': excel_row,
                         'text': clean(raw['response_text']), 'raw_text': raw['response_text'],
                         'email_group': unicodedata.normalize('NFC', str(raw['Email address'])).strip().casefold(),
                         'question': clean(raw['soal']), 'reference': clean(raw['jawaban_referensi']),
                         'response_number': str(raw['response_number']), 'score': int(score)})
    finally:
        book.close()
    if cfg.get('expected_rows') is not None and len(rows) != cfg['expected_rows']:
        raise ValueError(f"Diharapkan {cfg['expected_rows']} baris, ditemukan {len(rows)}; tidak dipotong otomatis.")
    if not rows:
        raise ValueError('Dataset kosong')
    parent = list(range(len(rows)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        parent[find(b)] = find(a)

    email_first, duplicate_sets = {}, {}
    for i, row in enumerate(rows):
        email = row['email_group']
        if email in email_first:
            union(email_first[email], i)
        email_first[email] = i
        # Normalisasi ini HANYA audit kesamaan, bukan teks masukan encoder.
        normalized = ' '.join(row['text'].casefold().split())
        key = hashlib.sha256(normalized.encode()).hexdigest()
        duplicate_sets.setdefault(key, []).append(i)
    for indices in duplicate_sets.values():
        for i in indices[1:]:
            union(indices[0], i)
    roots = {root: i for i, root in enumerate(sorted({find(i) for i in range(len(rows))}))}
    groups = np.asarray([roots[find(i)] for i in range(len(rows))], dtype=int)
    y = np.asarray([r['score'] for r in rows], dtype=float)
    summary = {'scope': SCOPE, 'rows': len(rows), 'target_columns': ['rater'], 'model_outputs': 1,
               'score_range_observed': [float(y.min()), float(y.max())],
               'score_range_declared': [cfg['score_min'], cfg['score_max']],
               'scale_basis': '0–100 mengikuti notebook unggahan; deskriptor rubrik guru tidak tersedia.',
               'email_groups': len(email_first), 'connected_groups': len(roots),
               'questions': len({r['question'] for r in rows}),
               'duplicate_sets': sum(len(v) > 1 for v in duplicate_sets.values()),
               'cross_email_duplicate_sets': sum(len({rows[i]['email_group'] for i in v}) > 1 for v in duplicate_sets.values()),
               'source_sha256': source_hash,
               'input_features': ['soal','response_text','jawaban_referensi'],
               'grouping_note': 'Email dipakai sebagai kelompok pencegah kebocoran; tidak membuktikan identitas terverifikasi. Duplikat lintas email disatukan. Semua skenario memakai soal/referensi dan jawaban. Email hanya kelompok; nilai guru hanya target, bukan fitur encoder.'}
    return Bundle(rows, y, groups, source_hash, summary)


def run_folder(cfg, bundle):
    code_hash = NOTEBOOK_CODE_SHA256
    semantic = {k: v for k, v in cfg.items() if k not in ('dataset', 'output', 'trace_position', 'vector_source', 'new_answer', 'focus_word', 'new_question', 'new_reference')}
    key = hashlib.sha256(json.dumps({'source': bundle.source_sha256, 'config': semantic, 'code': code_hash}, sort_keys=True).encode()).hexdigest()[:16]
    folder = Path(cfg['output']) / key
    folder.mkdir(parents=True, exist_ok=True)
    dump(folder / 'effective_config.json', cfg)
    return folder

In [ ]:
%%aes
# Bantuan membaca tabel: penamaan kolom, angka, dan penjelasan kata berulang.
LABELS = {'row_id':'ID Jawaban','word':'Kata','covered':'Tercakup','word_index':'Urutan Kata',
          'token_id':'ID Token','token':'Token','position':'Posisi Token','window':'Bagian Teks',
          'attention_mask':'Dibaca Model','skor_guru':'Nilai Guru','prediksi_mentah':'Prediksi Skor',
          'kategori_salinan':'Skor Dibulatkan','di_luar_rentang':'Di Luar Rentang',
          'Skenario':'Skenario','Fold':'Fold','AC2':'Gwet AC2','RMSE':'RMSE (poin)',
          'out_of_range_fraction':'Di Luar Rentang (%)','Status':'Keterangan',
          'teks_model':'Jawaban Siswa','teks_asli':'Teks Asli','score':'Nilai Guru',
          'role':'Penggunaan Data','connected_group':'Kelompok Jawaban',
          'input_format':'Format Input','response_text':'Jawaban Siswa','rater':'Nilai Guru','soal':'Soal',
          'sentence_index':'Urutan Kalimat','text':'Kalimat','fold':'Fold','scenario':'Skenario',
          'encoder':'Encoder','vector':'Panjang Vektor','head':'Lapisan Prediksi','output':'Jumlah Skor',
          'used_for':'Digunakan untuk','id':'Kode','title':'Judul',
          'rows':'Jumlah Jawaban','file':'File Hasil','stage':'Isi File','status':'Keadaan',
          'sheets_and_shapes':'Tabel yang Tersimpan','epoch':'Epoch',
          'train_mse_scaled':'Loss Latih (MSE)','target_loss':'Nilai untuk Menghitung Loss',
          'best_epoch_so_far':'Epoch Terbaik Sejauh Ini','epochs_without_improvement':'Epoch Tanpa Perbaikan',
          'selisih_mentah':'Selisih Prediksi − Guru','berubah':'Teks Dibersihkan?',
          'token_index':'Urutan Token','start':'Awal Teks','end':'Akhir Teks',
          'token_indices':'Urutan Potongan Token','token_ids':'ID Potongan Token',
          'covered_tokens':'Token yang Terbaca','total_tokens':'Jumlah Token',
          'window_coverage':'Jumlah Bagian yang Membaca Token',
          'occurrence_indices':'Posisi Kata yang Dibandingkan','same_token_ids':'ID Token Sama?',
          'initial_cosine':'Kemiripan Sebelum Konteks (Cosine)',
          'initial_max_abs_difference':'Selisih Terbesar Sebelum Konteks',
          'contextual_cosine':'Kemiripan Setelah Konteks (Cosine)',
          'contextual_max_abs_difference':'Selisih Terbesar Setelah Konteks',
          'contextual_allclose_1e-6':'Vektor Setelah Konteks Sama?',
          'undefined_reasons':'Alasan Metrik Tidak Tersedia','authors':'Penulis','year':'Tahun',
          'url':'Tautan','limitation':'Batas Penerapan','library':'Pustaka','version':'Versi',
          'name':'Pengaturan','value':'Nilai','field':'Keterangan','target':'Jenis Nilai',
          'Posisi Kata 1':'Posisi Kata A','Posisi Kata 2':'Posisi Kata B',
          'ID Token Sama':'ID Token Sama?','Vektor Awal Sama':'Vektor Sebelum Konteks Sama?',
          'Vektor Akhir Sama':'Vektor Setelah Konteks Sama?',
          'Kemiripan Vektor Akhir':'Kemiripan Setelah Konteks (Cosine)','Apa Artinya':'Penjelasan'}

REPEATED_COLUMNS = ['Model','Kata','Posisi Kata A','Posisi Kata B','ID Token Sama?',
                    'Vektor Sebelum Konteks Sama?','Vektor Setelah Konteks Sama?',
                    'Kemiripan Setelah Konteks (Cosine)','Penjelasan']

MODELS = {'S1':'RoBERTa','S2':'BGE-M3','S3':'Fusion RoBERTa + BGE-M3'}

ROLES = {'development':'Data latih (termasuk validasi silang)','test':'Data uji',
         'train':'Latih','validation':'Validasi'}

REPEATED_NOTES = [
    'Satu baris membandingkan kata yang sama pada dua posisi di jawaban contoh. Posisi dihitung mulai 1.',
    'Sebelum konteks = vektor dari tabel kosakata model. Setelah konteks = vektor setelah model membaca posisi dan kata di sekitarnya.',
    'Cosine mengukur kemiripan arah vektor: mendekati 1 berarti sangat mirip. Nilai yang terlihat 1 setelah pembulatan belum membuktikan seluruh angka vektornya sama.',
    'Kolom Vektor ... Sama? memeriksa selisih setiap komponen, dengan batas 0,000001. Ini batas perbandingan angka komputer, bukan tolerance skor AES.',
]

METRIC_NOTES = [
    'S1 = RoBERTa; S2 = BGE-M3; S3 = fusion keduanya. RMSE memakai satuan poin skor: lebih kecil berarti kesalahan lebih kecil.',
    'Pearson menunjukkan hubungan perubahan skor; QWK dan Gwet AC2 menunjukkan kesepakatan skor kategori. Korelasi tinggi saja belum menjamin prediksi mendekati nilai guru.',
    'Tanda — berarti hasil tidak tersedia, bukan nol. Baca keempat metrik bersama-sama.',
]

REPEATED_HEADERS=REPEATED_COLUMNS

LABELS.update({'input_segment':'Bagian Input','question_tokens':'Token Soal Awal',
    'question_kept':'Token Soal Dibaca','reference_tokens':'Token Referensi Awal',
    'reference_kept':'Token Referensi Dibaca','question_truncated':'Soal Terpotong?',
    'reference_truncated':'Referensi Terpotong?','context_tokens_kept':'Token Konteks Dibaca',
    'content_tokens':'Token Jawaban','capacity':'Kapasitas Jawaban per Window',
    'stride':'Langkah Window','starts':'Posisi Awal Window','windows':'Jumlah Window',
    'pair_special_tokens':'Token Khusus Pasangan','answer_tokens_uncovered':'Token Jawaban Tidak Tercakup',
    'context_max_tokens':'Batas Token Konteks Semua Encoder','question_max_tokens':'Batas Token Soal Semua Encoder'})

def friendly_header(name):
    match = re.fullmatch(r'v_(\d+)', str(name))
    return 'Komponen '+str(int(match[1])+1) if match else LABELS.get(name, str(name).replace('_',' '))


def friendly_value(name, value):
    if hasattr(value,'item'):
        try:value=value.item()
        except ValueError:pass
    if name == 'row_id' and isinstance(value,str):
        match = re.fullmatch(r'row_(\d+)',value)
        return f'J{int(match[1]):03d}' if match else value
    if name == 'role':
        return ROLES.get(value,value)
    if name == 'status':
        return {'ACTUAL EXECUTION':'Dihasilkan saat kode dijalankan',
                'DESIGN / CONFIGURATION':'Rancangan dan pengaturan model'}.get(value,value)
    if name == 'attention_mask' and value in (0,1):
        return 'Ya' if value else 'Tidak'
    if name == 'out_of_range_fraction' and isinstance(value,(int,float)):
        return value*100
    if name in ('word_index','sentence_index','token_index','position','window') and isinstance(value,(int,float)):
        return value+1
    if isinstance(value,bool):
        return 'Ya' if value else 'Tidak'
    return value


def repeated_word_rows(records,model):
    for row in records:
        positions=row.get('occurrence_indices')
        if isinstance(positions,str):positions=json.loads(positions)
        positions=positions or []
        same_ids=row.get('same_token_ids')
        initial=row.get('initial_max_abs_difference')
        same_initial=None if initial is None else abs(initial)<=1e-6
        same_context=row.get('contextual_allclose_1e-6')
        if same_ids is False:explanation='Kata sama, tetapi potongan token berbeda. Perhatikan tokenisasinya sebelum membandingkan vektor.'
        elif same_ids is None:explanation='Informasi kesamaan ID token belum tersedia.'
        elif same_context is None:explanation='Hasil perbandingan vektor akhir belum tersedia.'
        elif same_context:explanation='ID token sama; vektor akhir juga sama dalam toleransi numerik 0,000001.'
        else:explanation='ID token sama, tetapi vektor setelah membaca konteks berbeda. Posisi dan kata di sekitarnya dapat memengaruhi vektor.'
        label={'roberta':'RoBERTa','bge':'BGE-M3'}.get(model,model)
        yield [label,row.get('word'),positions[0]+1 if positions else None,positions[1]+1 if len(positions)>1 else None,
               friendly_value('',same_ids),friendly_value('',same_initial),friendly_value('',same_context),
               row.get('contextual_cosine'),explanation]

def preview_table(headers, records, title='', row_count=None, max_rows=8, max_columns=8):
    """Bentuk preview tanpa mengubah DataFrame/Excel sumber atau menghitung ulang model."""
    headers=list(headers)
    records=list(itertools.islice(records,max_rows))
    count=len(records) if row_count is None else row_count
    notes=[]
    original_columns=len(headers)
    if 'occurrence_indices' in headers:
        values=list(repeated_word_rows(records,''))
        # Nama model hanya ditampilkan bila memang diberikan oleh pemanggil.
        headers=REPEATED_HEADERS[1:]
        values=[r[1:] for r in values]
        notes=REPEATED_NOTES
        empty='Belum ada pasangan kata berulang yang bisa dibandingkan. Ini bukan error; lihat tabel vektor kata di atas.'
    elif {'file','stage','sheets_and_shapes'}.issubset(headers):
        # Dictionary ukuran tabel adalah rincian internal, bukan isi laporan belajar.
        headers=['File Hasil','Isi File','Keadaan']
        values=[[r.get('file'),r.get('stage'),friendly_value('status',r.get('status'))] for r in records]
        notes=['Daftar ini menunjukkan file yang sudah dibuat. Pilih lima laporan utama pada bagian unduhan untuk membaca hasilnya.']
        empty='Belum ada file hasil yang tersimpan.'
    else:
        selected=headers[:max_columns]
        values=[[friendly_value(key,row.get(key)) for key in selected] for row in records]
        headers=[friendly_header(key) for key in selected]
        empty='Belum ada baris hasil pada tahap ini. Tidak ada angka pengganti yang ditampilkan.'
        if 'Vektor Setelah Konteks Sama?' in headers or 'Vektor Akhir Sama' in headers:
            notes=REPEATED_NOTES
            empty='Belum ada pasangan kata berulang yang bisa dibandingkan. Ini bukan error; lihat tabel vektor kata di atas.'
        elif {'RMSE','Pearson','QWK','AC2'}.issubset(selected) or 'RMSE (poin)' in headers:
            notes=METRIC_NOTES
        elif 'prediksi_mentah' in selected or 'Prediksi S1' in headers:
            notes=['Prediksi Skor adalah nilai yang diberikan model. Nilai Guru adalah acuan jika tersedia.',
                   'Skor Dibulatkan adalah salinan untuk metrik kategori; prediksi asli tetap disimpan. Di Luar Rentang berarti prediksi melewati batas skor yang ditetapkan.']
        elif 'token_id' in selected or 'ID Token' in headers:
            notes=['Token adalah potongan teks yang dibaca model; satu kata bisa terpecah menjadi beberapa token. ID Token adalah nomor dalam kosakata model, bukan nilai siswa.',
                   'Posisi Token dan Bagian Teks dihitung mulai 1. Token khusus seperti <s> dan </s> menandai batas input.']
        components=sum(bool(re.fullmatch(r'v_\d+|Komponen \d+',str(h))) for h in selected)
        if components:
            notes=['Satu baris berisi satu vektor. Komponen 1, 2, dan seterusnya adalah angka penyusun vektor, bukan skor penilaian.',
                   'Tabel ini hanya memperlihatkan beberapa komponen pertama. Seluruh komponen tersimpan di Excel. Komponen yang sama nomornya pada RoBERTa dan BGE-M3 tidak memiliki arti yang otomatis sama.']
    return {'title':title,'columns':headers,'rows':values,'shape':[count,original_columns],
            'notes':notes,'empty_message':empty}


def cell_text(value):
    if value is None:return '—'
    if isinstance(value,bool):return 'Ya' if value else 'Tidak'
    if isinstance(value,float):
        return format(value,'.8g') if math.isfinite(value) else '—'
    if isinstance(value,dict):
        return '; '.join(f'{friendly_header(k)}: {cell_text(v)}' for k,v in value.items()) or '—'
    if isinstance(value,(list,tuple)):
        return ', '.join(cell_text(x) for x in value) or '—'
    return str(value)


def render_table(event):
    """HTML escapé dengan warna eksplisit agar terbaca pada Colab terang maupun gelap."""
    esc=lambda value:html.escape(cell_text(value))
    parts=['<section style="background:#f8fafc!important;color:#172035!important;padding:18px;border:1px solid #94a3b8;border-radius:12px;margin:12px 0">',
           '<h3 style="color:#172035!important;margin:0 0 10px">'+esc(event.get('title',''))+'</h3>']
    for note in event.get('notes',[]):parts.append('<p style="color:#334155!important;margin:6px 0">'+esc(note)+'</p>')
    if not event['rows']:
        parts.append('<p>'+esc(event.get('empty_message','Belum ada hasil.'))+'</p>')
    else:
        parts.append(f'<p style="color:#475569!important">Ditampilkan {len(event["rows"])} dari {event["shape"][0]} baris. Angka pada layar diringkas; nilai lengkap ada di Excel.</p>')
        th='background:#172035!important;color:#fff!important;padding:10px;border:1px solid #64748b;text-align:left'
        td='background:#f8fafc!important;color:#172035!important;padding:10px;border:1px solid #cbd5e1;vertical-align:top;min-width:90px;max-width:460px;white-space:normal;overflow-wrap:anywhere'
        parts.append('<div style="max-height:420px;overflow:auto"><table style="border-collapse:collapse"><thead><tr>')
        parts.extend('<th style="'+th+'">'+esc(x)+'</th>' for x in event['columns'])
        parts.append('</tr></thead><tbody>')
        for row in event['rows']:
            parts.append('<tr>'+''.join('<td style="'+td+'">'+esc(x)+'</td>' for x in row)+'</tr>')
        parts.append('</tbody></table></div>')
    parts.append('</section>')
    return ''.join(parts)

# Memeriksa nama GPU, status T4, dan VRAM; Colab berhenti sebelum training jika GPU belum aktif.
def check_gpu(require_gpu=False):
    """Periksa hardware yang benar-benar tersedia, tanpa memuat model pretrained."""
    available = torch.cuda.is_available()
    if not available:
        if require_gpu:
            raise RuntimeError('GPU belum aktif. Pilih Runtime → Change runtime type → T4 GPU, '
                               'lalu jalankan notebook dari atas. Training Colab tidak dimulai di CPU.')
        return {'name': 'CPU', 'cuda_available': False, 'is_t4': False, 'memory_gib': None}
    name = torch.cuda.get_device_name(0)
    return {'name': name, 'cuda_available': True, 'is_t4': 'T4' in name.upper(),
            'memory_gib': round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2)}


In [ ]:
%%aes
bundle = read_dataset(cfg)
output = run_folder(cfg, bundle)
show(pd.DataFrame([bundle.summary]), 'Audit dataset asli')
show(pd.DataFrame([dict(row_id=r['row_id'], soal=r['question'], referensi=r['reference'],
                       respons=r['text'], rater=r['score']) for r in bundle.rows]), 'Tiga input dan satu label')
print('Sumber dataset SHA256:',bundle.source_sha256,flush=True)

## 4 · Split dan lima fold yang dapat dicocokkan

GroupShuffleSplit seed2026,128kandidat; pilih proporsi jumlah baris terdekat80:20 tanpa melihat label. GroupKFold5 hanya di development. Resep ini sama dengan keluarga E21–E32; keanggotaan aktual wajib diverifikasi ketika membandingkan ZIP, terutama jika file dataset berubah.

Pada dataset300 yang sama, targetnya development240 dan test60. Notebook menampilkan jumlah **aktual**. Test ini dipakai ulang untuk perbandingan historis dan bukan holdout baru yang belum pernah dilihat dalam penelitian.

In [ ]:
%%aes
def make_split(bundle, cfg):
    indices = np.arange(len(bundle.rows))
    splitter = GroupShuffleSplit(n_splits=cfg['split_candidates'], test_size=cfg['test_fraction'], random_state=cfg['split_seed'])
    # Pilih kedekatan JUMLAH BARIS dengan target 80:20, bukan skor/hasil model.
    # GroupShuffleSplit membagi kelompok; ukuran kelompok tidak selalu sama.
    candidates = list(splitter.split(indices, groups=bundle.groups))
    development, test = min(candidates, key=lambda pair: abs(len(pair[1])/len(indices)-cfg['test_fraction']))
    if len(set(bundle.groups[development])) < cfg['folds']:
        raise ValueError('Kelompok training tidak cukup untuk CV yang diminta.')
    folds = []
    for number, (a, b) in enumerate(GroupKFold(n_splits=cfg['folds']).split(development, groups=bundle.groups[development]), 1):
        training, validation = development[a], development[b]
        assert not set(bundle.groups[training]) & set(bundle.groups[validation])
        folds.append({'fold': number, 'train': training.tolist(), 'validation': validation.tolist()})
    assert not set(bundle.groups[development]) & set(bundle.groups[test])
    manifest = {'scope': SCOPE, 'source_sha256': bundle.source_sha256,
                'development': development.tolist(), 'test': test.tolist(), 'folds': folds,
                'development_rows': len(development), 'test_rows': len(test),
                'development_groups': len(set(bundle.groups[development])),
                'test_groups': len(set(bundle.groups[test])),
                'target_test_row_fraction': cfg['test_fraction'],
                'candidate_group_fraction': cfg['test_fraction'],
                'split_candidates': cfg['split_candidates'],
                'selection_uses_labels': False,
                'actual_test_row_fraction': len(test)/len(indices),
                'policy': 'Holdout kelompok: pilih kandidat deterministik terdekat target proporsi baris berdasarkan ukuran kelompok saja, tanpa label/hasil model. CV hanya di development. Email/duplikat tidak menyeberang. Mengukur kelompok baru pada pertanyaan ini, bukan unseen-prompt.',
                'test_used_for_selection': False}
    manifest['split_sha256'] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    return manifest


def contexts_from_bundle(bundle):
    """Ambil soal/referensi tanpa membaca target skor guru."""
    contexts = []
    for row in bundle.rows:
        question, reference = row.get('question'), row.get('reference')
        if not isinstance(question, str) or not question.strip():
            raise ValueError('Context pair memerlukan soal yang tidak kosong.')
        if not isinstance(reference, str) or not reference.strip():
            raise ValueError('Context pair memerlukan jawaban referensi yang tidak kosong.')
        contexts.append((clean(question), clean(reference)))
    return contexts

In [ ]:
%%aes
manifest = make_split(bundle, cfg)
dump(output/'split_manifest.json', manifest)
show(pd.DataFrame([dict(development=len(manifest['development']), test=len(manifest['test']),
                       folds=len(manifest['folds']), development_groups=manifest['development_groups'],
                       test_groups=manifest['test_groups'])]), 'Partisi aktual')
print('Identitas split:',manifest['split_sha256'],flush=True)

## 5 · Ekstraksi tiga teks dengan encoder frozen

Soal dan referensi membentuk konteks, disambung dengan window respons. BGE memakai `eval()`, `requires_grad=False`, `no_grad()`; skor guru tidak masuk encoder. Mean CLS antar-window→L2 menghasilkan1024komponen.

Cache mempunyai identitas data/resep/checksum. Budget konteks merupakan resep acuan yang dikendalikan untuk perbandingan, bukan klaim bahwa paper menetapkannya optimal untuk AES. Tidak menambah fitur baru.

In [ ]:
%%aes
def mean_by_essay(cls, mapping, n):
    sums=cls.new_zeros((n,cls.shape[-1])).index_add(0,mapping,cls)
    counts=cls.new_zeros(n).index_add(0,mapping,cls.new_ones(len(mapping)))
    if (counts==0).any(): raise ValueError('Esai tanpa window')
    return sums / counts[:,None]


def context_token_ids(context, tokenizer, cfg):
    """Sisakan anggaran untuk soal dan referensi sekaligus."""
    if not isinstance(context, (tuple, list)) or len(context) != 2:
        raise ValueError('Konteks harus pasangan (soal, jawaban referensi).')
    question, reference = context
    if not all(isinstance(t, str) and t.strip() for t in context):
        raise ValueError('Isi soal dan jawaban referensi sebelum memprediksi.')
    limit, qlimit = cfg['context_max_tokens'], cfg['question_max_tokens']
    if not 1 <= qlimit < limit:
        raise ValueError('Anggaran token soal harus lebih kecil daripada konteks.')
    # Prefix menjadi bagian input; nilai guru tidak pernah dimasukkan.
    qids = tokenizer.encode('Soal: '+question, add_special_tokens=False)
    rids = tokenizer.encode('\nJawaban referensi: '+reference, add_special_tokens=False)
    question_kept = qids[:qlimit]
    reference_kept = rids[:limit-len(question_kept)]
    return question_kept+reference_kept, {
        'question_tokens': len(qids), 'question_kept': len(question_kept),
        'reference_tokens': len(rids), 'reference_kept': len(reference_kept),
        'question_truncated': len(question_kept) < len(qids),
        'reference_truncated': len(reference_kept) < len(rids),
        'context_tokens_kept': len(question_kept)+len(reference_kept),
    }


def pair_content_windows(text, context, tokenizer, maximum=512, overlap=128, cfg=None):
    """Informasi dan window sama; E22 menambah separator resmi pasangan."""
    if cfg is None or cfg.get('input_format') not in ('single_sequence', 'context_pair'):
        raise ValueError('Pilih format input E21 atau E22 sebelum tokenisasi.')
    context_ids, context_info = context_token_ids(context, tokenizer, cfg)
    ids = tokenizer.encode(text, add_special_tokens=False)
    answer_prefix = tokenizer.encode('\nJawaban siswa:\n', add_special_tokens=False)
    if not ids:
        raise ValueError('Jawaban tidak menghasilkan token.')
    paired = cfg['input_format'] == 'context_pair'
    specials = tokenizer.num_special_tokens_to_add(pair=paired)
    # Cadangkan overhead terbesar untuk KEDUA eksperimen. Starts/cakupan sama.
    reserved = max(tokenizer.num_special_tokens_to_add(pair=False),
                   tokenizer.num_special_tokens_to_add(pair=True))
    capacity = maximum-len(context_ids)-len(answer_prefix)-reserved
    if not 0 <= overlap < capacity:
        raise ValueError('Konteks dan penanda jawaban menyisakan kapasitas di bawah overlap.')
    starts = [0]
    while starts[-1]+capacity < len(ids):
        starts.append(starts[-1]+capacity-overlap)
    chunks, answer_positions, segments = [], [], []
    for start in starts:
        answer = ids[start:start+capacity]
        second = answer_prefix+answer
        if paired:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids, second)
            layout = tokenizer.build_inputs_with_special_tokens([-1]*len(context_ids),
                     [-3]*len(answer_prefix)+[-2]*len(answer))
        else:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids+second)
            layout = tokenizer.build_inputs_with_special_tokens(
                     [-1]*len(context_ids)+[-3]*len(answer_prefix)+[-2]*len(answer))
        positions = [i for i, value in enumerate(layout) if value == -2]
        if (len(prepared) != len(layout) or len(prepared) > maximum
                or len(prepared)-len(context_ids)-len(second) != specials
                or [prepared[i] for i in positions] != answer):
            raise ValueError('Template tokenizer tidak konsisten dengan format input.')
        chunks.append({'input_ids':prepared, 'attention_mask':[1]*len(prepared)})
        answer_positions.append(positions)
        segments.append(['Soal/referensi' if v == -1 else 'Jawaban siswa' if v == -2
                         else 'Penanda jawaban' if v == -3 else 'Token khusus' for v in layout])
    stats = {**context_info, 'input_format':cfg['input_format'], 'content_tokens':len(ids),
             'capacity':capacity, 'stride':capacity-overlap, 'starts':starts,
             'windows':len(chunks), 'pair_special_tokens':specials,
             'reserved_special_tokens':reserved, 'answer_prefix_tokens':len(answer_prefix),
             'answer_positions':answer_positions, 'segments':segments, 'answer_tokens_uncovered':0}
    return chunks, stats


def pair_batch_windows(texts, contexts, tokenizer, device='cpu', maximum=512, overlap=128, cfg=None):
    """Susun batch pasangan tanpa menukar konteks antarjawaban."""
    if len(texts) != len(contexts) or not texts:
        raise ValueError('Jumlah konteks harus sama dengan jumlah jawaban.')
    chunks, mapping, stats = [], [], []
    for index, (text, context) in enumerate(zip(texts, contexts)):
        values, detail = pair_content_windows(text, context, tokenizer, maximum, overlap, cfg)
        chunks.extend(values)
        mapping.extend([index]*len(values))
        stats.append(detail)
    batch = tokenizer.pad(chunks, padding=True, return_tensors='pt')
    return {key: value.to(device) for key, value in batch.items()}, torch.tensor(mapping, device=device), stats


def paired_pretrained_values(bundle, cfg, role):
    """Mean CLS antar-window; BGE dinormalisasi L2 setelah agregasi."""
    device = device_name(cfg)
    model, tok = encoder(cfg,role).to(device), tokenizer(cfg,role)
    model.eval()
    vectors, window_vectors, ids, coverage = [], [], [], []
    with torch.no_grad():
        for index, (text,context) in enumerate(zip(bundle.texts,contexts_from_bundle(bundle))):
            windows,mapping,details = pair_batch_windows([text],[context],tok,device,
                cfg[role+'_max_length'],cfg[role+'_overlap'],cfg)
            # Satu jawaban diproses bertahap agar jumlah window tidak meledakkan memori.
            cls_parts=[]
            for start in range(0,len(mapping),cfg['bge_batch']):
                part={k:v[start:start+cfg['bge_batch']] for k,v in windows.items()}
                cls_parts.append(model(**part).last_hidden_state[:,0,:].float())
            cls=torch.cat(cls_parts,dim=0)
            value=mean_by_essay(cls,mapping,1)
            if role=='bge':value=torch.nn.functional.normalize(value,dim=1)
            vectors.append(value.cpu().numpy())
            window_vectors.append(cls.cpu().numpy())
            row_id=bundle.rows[index].get('row_id',f'inference_{index}')
            ids.extend(f'{row_id}_window_{j}' for j in range(len(cls)))
            coverage.append({k:v for k,v in details[0].items() if k not in ('answer_positions','segments')})
            if (index+1)%25==0 or index+1==len(bundle.rows):
                print(f'{role}: tiga input {index+1}/{len(bundle.rows)} jawaban selesai.',flush=True)
    values=np.concatenate(vectors).astype(np.float32)
    if role=='bge' and any(p.requires_grad for p in model.parameters()):
        raise ValueError('Encoder BGE harus frozen.')
    del model,tok
    release()
    return values,np.concatenate(window_vectors),ids,coverage

In [ ]:
%%aes
def device_name(cfg):
    requested = cfg['device']
    if requested == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA diminta tetapi tidak tersedia; pilih GPU di Colab atau device=cpu.')
    return ('cuda' if torch.cuda.is_available() else 'cpu') if requested == 'auto' else requested


def release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class ExpectedPoolerNotice(logging.Filter):
    """Hanya saring pemberitahuan dua bobot pooler yang memang tidak dipakai."""
    def filter(self, record):
        import ast
        import re
        message = record.getMessage()
        if not message.startswith('Some weights of the model checkpoint at ') or 'were not used when initializing' not in message:
            return True
        match = re.search(r"\['[^\n]+?\]", message)
        if match is None:
            return True
        try:
            keys = ast.literal_eval(match.group())
        except (ValueError, SyntaxError):
            return True
        return set(keys) != {'pooler.dense.bias', 'pooler.dense.weight'}


def encoder(cfg, role):
    # Ambil nama checkpoint dan revision model yang dipilih: roberta atau bge.
    if role != 'bge':
        raise ValueError('Eksperimen ini hanya memuat BGE frozen.')
    spec = cfg[role]
    logger = logging.getLogger('transformers.modeling_utils')
    notice = ExpectedPoolerNotice()
    logger.addFilter(notice)
    try:
        # Muat arsitektur dan bobot pretrained. Ini baru inisialisasi encoder, belum latihan memakai nilai guru.
        result = AutoModel.from_pretrained(spec['checkpoint'], revision=spec['revision'], add_pooling_layer=False)
    finally:
        logger.removeFilter(notice)
    # Periksa panjang vektor: RoBERTa 768 angka, BGE-M3 1024 angka.
    expected = 768 if role == 'roberta' else 1024
    if result.config.hidden_size != expected:
        raise ValueError(f'{role}: dimensi checkpoint tidak cocok dengan {expected}.')
    # Parameter RoBERTa ikut diperbarui; parameter BGE tidak ikut diperbarui.
    # RoBERTa boleh belajar; bobot encoder BGE tidak diperbarui.
    result.requires_grad_(False)
    return result


def tokenizer(cfg, role, fast=False):
    # Muat tokenizer dari checkpoint yang sama agar ID token cocok dengan kosakata model.
    result = AutoTokenizer.from_pretrained(cfg[role]['checkpoint'], revision=cfg[role]['revision'],
                                           use_fast=fast if role == 'roberta' else True)
    if role == 'roberta' and not fast and result.num_special_tokens_to_add(False) != 2:
        raise ValueError('RoBERTa slow tokenizer harus menambahkan dua special token.')
    return result

In [ ]:
%%aes
def bge_features(bundle, cfg, output):
    """Cache berubah jika soal, referensi, jawaban, atau resep token berubah."""
    contexts=contexts_from_bundle(bundle)
    contract={'model':cfg['bge'], 'texts':bundle.texts, 'contexts':contexts,
              'maximum':cfg['bge_max_length'],'overlap':cfg['bge_overlap'],
              'context_max_tokens':cfg['context_max_tokens'],'question_max_tokens':cfg['question_max_tokens'],
              'policy':'three_inputs_mean_CLS_then_L2_v2', 'input_format':cfg['input_format']}
    key=hashlib.sha256(json.dumps(contract,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:20]
    cache=Path(cfg['output'])/'cache'/f'bge_three_inputs_{key}.npz'
    metadata=cache.with_suffix('.json')
    if cache.exists() and metadata.exists():
        proof=json.loads(metadata.read_text())
        if proof['file_sha256']!=hashlib.sha256(cache.read_bytes()).hexdigest():
            raise ValueError('Cache BGE berubah; tidak dipakai.')
        with np.load(cache,allow_pickle=False) as data:values=data['features']
        reused=True
    else:
        values,_,_,coverage=paired_pretrained_values(bundle,cfg,'bge')
        cache.parent.mkdir(parents=True,exist_ok=True)
        np.savez_compressed(cache,features=values)
        proof={'model':cfg['bge'],'shape':list(values.shape),'frozen':True,
               'pooling':'mean paired-window CLS then L2','max_length':cfg['bge_max_length'],
               'feature_source':'soal + jawaban_referensi + response_text; no teacher score', 'input_format':cfg['input_format'],
               'answer_tokens_uncovered':sum(s['answer_tokens_uncovered'] for s in coverage),
               'context_truncated_rows':sum(s['question_truncated'] or s['reference_truncated'] for s in coverage),
               'file_sha256':hashlib.sha256(cache.read_bytes()).hexdigest()}
        dump(metadata,proof)
        reused=False
    if values.shape!=(len(bundle.rows),1024) or not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values,axis=1),1,atol=1e-4):
        raise ValueError('Fitur BGE shape/normalisasi tidak sah.')
    dump(Path(output)/'bge_encoding.json',{**proof,'reused':reused})
    return torch.from_numpy(values)
bge = bge_features(bundle, cfg, output)
X = bge.numpy().astype(np.float32)
np.save(output/'bge_frozen.npy', X)
show(pd.DataFrame([dict(shape=str(X.shape), norm_min=float(np.linalg.norm(X,axis=1).min()),
    norm_max=float(np.linalg.norm(X,axis=1).max()), encoder_trained=False)]), 'Fitur tiga input untuk Ridge')
print('BGE selesai; model encoder sudah dilepas dari memori. Selanjutnya Ridge CPU float64.',flush=True)

## 6 · Metrik tanpa menyembunyikan error

RMSE, MAE, bias, median/p90/p95/max absolute error serta jumlah prediksi dalam±1,±2,±5,±10poin memakai skor kontinu asli tanpa clipping. QWK/Gwet AC2 menggunakan salinan kategori0–100, tie.5ke bawah; Pearson atau agreement undefined diberi alasan.

RMSE menekankan error besar. Hitungan toleransi menghitung jumlah jawaban dekat. Keduanya diperlukan dan tidak otomatis memilih model yang sama.

In [ ]:
%%aes
def gwet_ac2(teacher, predicted, categories):
    a=np.asarray(teacher);b=np.asarray(predicted);c=np.asarray(categories,dtype=float)
    if len(a)!=len(b) or len(a)<2: raise ValueError('AC2 perlu minimal dua pasangan')
    if not np.isin(a,c).all() or not np.isin(b,c).all(): raise ValueError('Kategori AC2 tidak sah')
    q=len(c); weights=1-((c[:,None]-c[None,:])/(c.max()-c.min()))**2
    ai=np.searchsorted(c,a); bi=np.searchsorted(c,b)
    pa=weights[ai,bi].mean()
    pi=(np.bincount(ai,minlength=q)+np.bincount(bi,minlength=q))/(2*len(a))
    pe=weights.sum()*np.sum(pi*(1-pi))/(q*(q-1))
    return float((pa-pe)/(1-pe))

def metrics(teacher, prediction, cfg):
    y, p = np.asarray(teacher, dtype=float), np.asarray(prediction, dtype=float)
    if y.ndim != 1 or y.shape != p.shape or len(y) == 0:
        raise ValueError('Metrik satu skor memerlukan dua vektor [N] yang sama dan tidak kosong.')
    if not np.isfinite(y).all() or not np.isfinite(p).all():
        raise ValueError('Label/prediksi tidak finite.')
    cats = np.arange(cfg['score_min'], cfg['score_max']+1)
    if not np.isin(y, cats).all():
        raise ValueError('Skor guru harus kategori integer pada rentang yang dinyatakan.')
    # Buat SALINAN skor sah untuk agreement; prediksi mentah tetap utuh.
    clipped = np.clip(p, cfg['score_min'], cfg['score_max'])
    # Tie .5 memilih kategori lebih rendah, sama dengan kontrak konversi proyek.
    ordinal = cats[np.abs(clipped[:, None]-cats[None, :]).argmin(axis=1)]
    result = {'RMSE': float(np.sqrt(np.mean((y-p)**2))), 'Pearson': None,
              'QWK': None, 'AC2': None, 'undefined_reasons': {},
              'out_of_range_fraction': float(np.mean((p < cats[0]) | (p > cats[-1])))}
    if len(y) > 1 and np.ptp(y) > 0 and np.ptp(p) > 0:
        result['Pearson'] = float(pearsonr(y, p).statistic)
    else:
        result['undefined_reasons']['Pearson'] = 'Perlu minimal dua pasangan dan kedua vektor tidak konstan.'
    if len(y) > 1:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            # QWK memakai jarak kategori kuadratik, bukan vektor embedding.
            qwk = cohen_kappa_score(y, ordinal, labels=cats, weights='quadratic')
        if np.isfinite(qwk):
            result['QWK'] = float(qwk)
        else:
            result['undefined_reasons']['QWK'] = 'Expected disagreement nol; undefined, bukan nol.'
        # AC2 memakai kategori yang sama dengan QWK dan koreksi peluang berbeda.
        result['AC2'] = gwet_ac2(y, ordinal, cats)
    else:
        result['undefined_reasons'].update(QWK='Perlu minimal dua pasangan.', AC2='Perlu minimal dua pasangan.')
    return result

## 7 · Ridge dan pemilihan alpha yang terpisah

Untuk setiap outer fold, alpha dipilih pada empat blok fold lain; outer validation tidak ikut seleksi. Hasil outer digabung menjadi nested OOF development. Setelah itu, alpha final dipilih melalui lima fold development dan model baru dipasang pada seluruh development.

Kontrak alpha final dan koefisien disimpan **sebelum** menghitung skor test. Nilai CV yang digunakan memilih alpha bukan skor evaluasi independen. Matriks fitur disimpan float32 seperti acuan, perhitungan Ridge float64 untuk kestabilan numerik.

In [ ]:
%%aes
def write_readout_workbook(path, sheets):
    # Strings are literal text, never spreadsheet formulas from uploaded essays.
    book = Workbook(write_only=True)
    for title, data in sheets.items():
        frame = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
        sheet = book.create_sheet(title[:31])
        sheet.append([str(v) for v in frame.columns])
        for record in frame.itertuples(index=False, name=None):
            cells = []
            for value in record:
                if isinstance(value, (list, dict, tuple)):
                    value = json.dumps(value, ensure_ascii=False)
                if isinstance(value, np.generic):
                    value = value.item()
                if isinstance(value, float) and not math.isfinite(value):
                    value = None
                cell = WriteOnlyCell(sheet, value=value)
                if isinstance(value, str):
                    cell.data_type = 's'
                cells.append(cell)
            sheet.append(cells)
    book.save(path)

In [ ]:
%%aes
"""Embedded runtime for the standalone three-input frozen-BGE Ridge experiment.

Imports and the unchanged dataset/encoder/metric helpers are embedded by the builder.
This module contains no dataset, scores, trained weights, or research results.
"""
from sklearn.linear_model import Ridge

RIDGE_BOOKS = ('01_Data_dan_Split.xlsx', '02_Vektor_BGE_dan_Model.xlsx',
               '03_Prediksi_OOF_dan_Test.xlsx', '04_Evaluasi_dan_Seleksi.xlsx',
               '05_Pengaturan_dan_Referensi.xlsx')
RIDGE_PAPERS = [
    dict(method='BGE-M3', title='BGE M3-Embedding: Multi-Lingual, Multi-Functionality, Multi-Granularity Text Embeddings Through Self-Knowledge Distillation',
         authors='Chen et al. (2024)', url='https://arxiv.org/abs/2402.03216',
         use='Dasar encoder; bukan bukti alpha terbaik atau jaminan performa AES.'),
    dict(method='Ridge', title='Ridge Regression: Biased Estimation for Nonorthogonal Problems',
         authors='Hoerl & Kennard (1970)', url='https://doi.org/10.1080/00401706.1970.10488634',
         use='Regresi linear dengan penalti L2; alpha tetap harus dipilih pada data development.'),
    dict(method='Seleksi dan evaluasi terpisah', title='On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation',
         authors='Cawley & Talbot (2010)', url='https://www.jmlr.org/papers/v11/cawley10a.html',
         use='Dasar pemisahan seleksi parameter dari evaluasi; nested CV untuk OOF prosedur seleksi.'),
]


def validate_ridge_inputs(bundle, X, manifest, cfg):
    X = np.asarray(X, dtype=np.float64)
    if X.shape != (len(bundle.rows), 1024) or not np.isfinite(X).all():
        raise ValueError('Ridge perlu fitur BGE finite [jumlah jawaban, 1024].')
    if not np.allclose(np.linalg.norm(X, axis=1), 1., atol=1e-4):
        raise ValueError('Fitur harus menggunakan L2 per jawaban yang sama dengan acuan.')
    alphas = cfg['ridge_alphas']
    if not alphas or len(set(alphas)) != len(alphas) or any(
            isinstance(a, bool) or not math.isfinite(a) or a <= 0 for a in alphas):
        raise ValueError('Alpha harus unik, positif, finite dan ditentukan sebelum evaluasi.')
    dev, test = set(manifest['development']), set(manifest['test'])
    if not dev or not test or dev & test or dev | test != set(range(len(X))):
        raise ValueError('Development dan test harus membagi seluruh baris tanpa overlap.')
    if len(dev) != len(manifest['development']) or len(test) != len(manifest['test']):
        raise ValueError('Indeks partisi duplikat.')
    if set(bundle.groups[list(dev)]) & set(bundle.groups[list(test)]):
        raise ValueError('Kelompok menyeberang development dan test.')
    covered = []
    for fold in manifest['folds']:
        tr, va = set(fold['train']), set(fold['validation'])
        if not tr or not va or tr & va or tr | va != dev:
            raise ValueError('Fold harus membagi development; test tidak boleh masuk fold.')
        if len(tr) != len(fold['train']) or len(va) != len(fold['validation']):
            raise ValueError('Indeks fold duplikat.')
        if set(bundle.groups[list(tr)]) & set(bundle.groups[list(va)]):
            raise ValueError('Kelompok menyeberang train dan validation.')
        covered.extend(fold['validation'])
    if len(manifest['folds']) < 3 or len(covered) != len(dev) or set(covered) != dev:
        raise ValueError('Setiap development wajib menjadi validation tepat sekali; minimal3fold.')
    return X


def select_ridge_alpha(X, y, allowed, blocks, alphas, stage):
    """Only labels in allowed participate; blocks are the existing grouped fold blocks."""
    allowed = set(allowed)
    pieces = [sorted(allowed & set(block)) for block in blocks]
    pieces = [piece for piece in pieces if piece]
    flat = [i for piece in pieces for i in piece]
    if len(pieces) < 2 or len(flat) != len(allowed) or set(flat) != allowed:
        raise ValueError('Blok inner/final CV tidak membagi train yang diizinkan tepat sekali.')
    search, details = [], []
    for alpha in alphas:
        squared, count = 0., 0
        for number, va in enumerate(pieces, 1):
            tr = sorted(allowed - set(va))
            model = Ridge(alpha=float(alpha), fit_intercept=True, solver='cholesky')
            model.fit(X[tr], y[tr])
            p = model.predict(X[va])
            sse = float(np.sum((p - y[va]) ** 2))
            squared += sse; count += len(va)
            details.append(dict(stage=stage, alpha=float(alpha), inner_fold=number,
                                train_rows=len(tr), validation_rows=len(va), RMSE=math.sqrt(sse / len(va))))
        search.append(dict(stage=stage, alpha=float(alpha), selection_RMSE=math.sqrt(squared / count),
                           selection_rows=count, selection_folds=len(pieces)))
    # Fixed tie rule: smaller alpha. No test metric is available to this function.
    selected = min(search, key=lambda r: (r['selection_RMSE'], r['alpha']))['alpha']
    for row in search:
        row['selected'] = row['alpha'] == selected
    return selected, search, details

In [ ]:
%%aes
def ridge_metric_row(partition, teacher, prediction, cfg, **extra):
    m = metrics(teacher, prediction, cfg)
    errors = np.asarray(prediction) - np.asarray(teacher)
    absolute = np.abs(errors)
    m.update(n=len(errors), MAE=float(absolute.mean()), bias=float(errors.mean()),
             median_absolute_error=float(np.median(absolute)), p90_absolute_error=float(np.quantile(absolute, .90)),
             p95_absolute_error=float(np.quantile(absolute, .95)), max_absolute_error=float(absolute.max()))
    for threshold in (1, 2, 5, 10):
        m['within' + str(threshold)] = int(np.sum(absolute <= threshold))
    reasons = json.dumps(m.pop('undefined_reasons'), ensure_ascii=False)
    return dict(partition=partition, **extra, **m, undefined_reasons=reasons)


def ridge_prediction_rows(bundle, indices, prediction, partition, fold, alpha):
    rows = []
    for i, p in zip(indices, prediction):
        y = float(bundle.y[i]); error = float(p - y)
        row = dict(row_id=bundle.rows[i]['row_id'], partition=partition, fold=fold,
                   model='BGE_M3_RIDGE_3INPUT', alpha=alpha, teacher=y, prediction=float(p),
                   error=error, absolute_error=abs(error))
        row.update({f'within{t}': abs(error) <= t for t in (1, 2, 5, 10)})
        rows.append(row)
    return rows

In [ ]:
%%aes
def run_ridge_three_input(bundle, features, manifest, cfg, output):
    X = validate_ridge_inputs(bundle, features, manifest, cfg)
    y = np.asarray(bundle.y, dtype=np.float64)
    blocks = [f['validation'] for f in manifest['folds']]
    dev = np.asarray(manifest['development'], dtype=int)
    test = np.asarray(manifest['test'], dtype=int)
    run = Path(output); run.mkdir(parents=True, exist_ok=True)
    evaluation, predictions, searches, details, outer = [], [], [], [], []
    oof = np.full(len(y), np.nan)
    for fold in manifest['folds']:
        tr, va, number = fold['train'], fold['validation'], fold['fold']
        alpha, grid, inner = select_ridge_alpha(X, y, tr, blocks, cfg['ridge_alphas'], f'OUTER_{number}')
        model = Ridge(alpha=alpha, fit_intercept=True, solver='cholesky').fit(X[tr], y[tr])
        p = model.predict(X[va]); oof[va] = p
        predictions.extend(ridge_prediction_rows(bundle, va, p, 'DEVELOPMENT_OOF', number, alpha))
        evaluation.append(ridge_metric_row('DEVELOPMENT_OUTER_FOLD', y[va], p, cfg, fold=number, alpha=alpha))
        evaluation.append(ridge_metric_row('TRAIN_OUTER_OPTIMISTIC', y[tr], model.predict(X[tr]), cfg, fold=number, alpha=alpha))
        outer.append(dict(fold=number, alpha=alpha, train_rows=len(tr), validation_rows=len(va)))
        searches.extend(grid); details.extend(inner)
        print(f'Ridge outer fold{number}: alpha={alpha}; validation={len(va)}', flush=True)
    if not np.isfinite(oof[dev]).all() or np.isfinite(oof[test]).any():
        raise ValueError('OOF development tidak lengkap atau berisi test.')
    evaluation.append(ridge_metric_row('DEVELOPMENT_POOLED_NESTED_OOF', y[dev], oof[dev], cfg, fold='ALL', alpha=None))
    # Final alpha chosen on development only; this score is for selection, not independent OOF performance.
    alpha, grid, inner = select_ridge_alpha(X, y, dev.tolist(), blocks, cfg['ridge_alphas'], 'FINAL_DEVELOPMENT_SELECTION')
    searches.extend(grid); details.extend(inner)
    feature_hash = hashlib.sha256(np.asarray(features, dtype=np.float32).tobytes()).hexdigest()
    membership = dict(development=manifest['development'], test=manifest['test'], folds=manifest['folds'])
    contract = dict(experiment='BGE_M3_RIDGE_3INPUT', dataset_sha256=bundle.source_sha256,
                    code_sha256=NOTEBOOK_CODE_SHA256, feature_sha256=feature_hash,
                    split_sha256=manifest['split_sha256'],
                    membership_sha256=hashlib.sha256(json.dumps(membership, sort_keys=True).encode()).hexdigest(),
                    selected_alpha=alpha, alpha_grid=cfg['ridge_alphas'],
                    selection_partition='development_only', selection_metric='pooled_RMSE',
                    selection_tie_rule='smaller_alpha', fit_intercept=True, solver='cholesky',
                    features='L2 per answer; no StandardScaler', head_dtype='float64',
                    target='rater 0–100, unscaled', test_used_for_selection=False,
                    evaluation_context='Retrospective shared historical holdout; not a new unseen test set.')
    dump(run / 'alpha_locked_before_test.json', contract)
    model = Ridge(alpha=alpha, fit_intercept=True, solver='cholesky').fit(X[dev], y[dev])
    model_path = run / 'ridge_final_model.npz'
    np.savez_compressed(model_path, coef=np.asarray(model.coef_, dtype=np.float64),
                        intercept=np.asarray(model.intercept_, dtype=np.float64), alpha=np.asarray(alpha))
    # Test is evaluated only after the selection contract and final coefficients have been saved.
    p = model.predict(X[test])
    predictions.extend(ridge_prediction_rows(bundle, test, p, 'TEST_HISTORICAL_HOLDOUT', 'FINAL_REFIT', alpha))
    evaluation.append(ridge_metric_row('TEST_HISTORICAL_HOLDOUT', y[test], p, cfg, fold='FINAL_REFIT', alpha=alpha))
    evaluation.append(ridge_metric_row('TRAIN_FINAL_OPTIMISTIC', y[dev], model.predict(X[dev]), cfg, fold='FINAL_REFIT', alpha=alpha))
    evaluation.append(ridge_metric_row('TEST_TRAIN_MEAN_BASELINE', y[test], np.full(len(test), y[dev].mean()), cfg, fold='FINAL_REFIT', alpha=None))
    result = dict(run_folder=str(run), contract=contract, evaluation=evaluation, predictions=predictions,
                  alpha_search=searches, inner_folds=details, outer_folds=outer,
                  test_evaluated=True, BGE_encoder_trained=False,
                  model_sha256=digest(model_path), model_file=model_path.name)
    dump(run / 'ridge_result.json', result)
    return result

In [ ]:
%%aes
def export_ridge_three_input(bundle, X, manifest, cfg, result):
    run = Path(result['run_folder']); destination = run / 'hasil_excel'; destination.mkdir(exist_ok=True)
    roles = {i: 'development' for i in manifest['development']}
    roles.update({i: 'test' for i in manifest['test']})
    records = [dict(row_id=r['row_id'], source_row=r['source_row'], soal=r['question'],
                    jawaban_referensi=r['reference'], response_text=r['text'], rater=float(bundle.y[i]),
                    response_number=r['response_number'], connected_group=int(bundle.groups[i]), role=roles[i])
               for i, r in enumerate(bundle.rows)]
    split = [dict(fold=f['fold'], role=role, row_id=bundle.rows[i]['row_id'])
             for f in manifest['folds'] for role in ('train', 'validation') for i in f[role]]
    guide = [dict(Keterangan=k, Nilai=v) for k, v in dict(
        model='BGE-M3 frozen → mean window CLS → L2 → Ridge; tiga input; satu rater',
        selection='Nested5fold OOF development; final alpha dari5fold development; final fit seluruh development',
        comparison='Bandingkan TEST dengan TEST S2; OOF dengan OOF yang protokolnya sepadan. Jangan campur OOF dan test.',
        historical='Holdout lama sudah dilihat pada eksperimen lain: perbandingan retrospektif, bukan test baru.',
        metrics='RMSE/MAE dan toleransi memakai prediksi kontinu tanpa clipping; skor agreement memakai salinan kategori.',
        training='Ridge bukan MLP; tanpa epoch/LR/dropout/early stopping; encoder tidak dilatih.',
        controls='E22 resep token context_pair berbeda; E29/E31 LR MLP5e-6. Catat kontrol S2 yang prediksinya identik.',
        join='Cocokkan ketiga teks, label, keanggotaan test dan fold; row_id saja tidak cukup.',
        privacy='Seluruh Excel dan model privat; jangan masukkan data siswa ke repository publik.').items()]
    frame = pd.DataFrame(X, columns=[f'Komponen {i+1}' for i in range(X.shape[1])])
    frame.insert(0, 'row_id', [r['row_id'] for r in bundle.rows])
    with np.load(run / result['model_file'], allow_pickle=False) as model:
        weights = pd.DataFrame(dict(component=np.arange(1, X.shape[1]+1), coefficient=model['coef']))
        intercept = float(model['intercept'])
    enc_path = run / 'bge_encoding.json'
    encoding = json.loads(enc_path.read_text()) if enc_path.is_file() else {'proof': 'not provided; technical fixture only'}
    write_readout_workbook(destination / RIDGE_BOOKS[0], dict(Panduan=guide, Data=records, Lima_Fold=split))
    write_readout_workbook(destination / RIDGE_BOOKS[1], dict(Panduan=guide, BGE_Frozen=frame, Ridge_Coef=weights,
        Model=[dict(intercept=intercept, alpha=result['contract']['selected_alpha'])],
        Bukti_Encoding=[dict(parameter=k, value=v) for k, v in encoding.items()]))
    write_readout_workbook(destination / RIDGE_BOOKS[2], dict(Panduan=guide,
        Prediksi_Development_OOF=[r for r in result['predictions'] if r['partition']=='DEVELOPMENT_OOF'],
        Prediksi_Test=[r for r in result['predictions'] if r['partition']=='TEST_HISTORICAL_HOLDOUT']))
    write_readout_workbook(destination / RIDGE_BOOKS[3], dict(Panduan=guide, Evaluasi=result['evaluation'],
        Alpha_Seleksi=result['alpha_search'], Inner_Fold=result['inner_folds'], Outer_Fold=result['outer_folds']))
    settings = dict(cfg, **result['contract'])
    write_readout_workbook(destination / RIDGE_BOOKS[4], dict(Panduan=guide,
        Pengaturan=[dict(parameter=k, value=v) for k, v in settings.items()], Referensi=RIDGE_PAPERS,
        Versi=[dict(library=k, version=version(k)) for k in ('torch','transformers','numpy','pandas','scikit-learn','openpyxl')]))
    dump(run / 'split_manifest.json', manifest)
    members = {name: destination / name for name in RIDGE_BOOKS}
    for name in ('ridge_final_model.npz', 'split_manifest.json', 'alpha_locked_before_test.json'):
        members[name] = run / name
    archive = run / 'Hasil_AES_300_BGE_M3_Ridge_Tiga_Input.zip'
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for name, path in members.items():
            z.write(path, name)
        z.writestr('manifest.json', json.dumps(dict(**result['contract'], test_evaluated=True,
            BGE_encoder_trained=False, files={name: digest(path) for name, path in members.items()}),
            ensure_ascii=False, indent=2))
    return dict(archive=str(archive), books=list(RIDGE_BOOKS), run_folder=str(run),
                selected_alpha=result['contract']['selected_alpha'], test_evaluated=True)

## 8 · Jalankan nested CV, final refit dan test

Tidak ada pemilihan seed terbaik, epoch terbaik, atau alpha terbaik berdasarkan test. Skor train ditandai **optimistis**; baseline test memakai rerata skor development saja.
Notebook tidak mengubah kode/hasil MLP lama dan tidak mengasumsikan Ridge pasti menang.

In [ ]:
%%aes
result = run_ridge_three_input(bundle, X, manifest, cfg, output)
print('Alpha final terkunci:', result['contract']['selected_alpha'], '| Test dievaluasi:', result['test_evaluated'], flush=True)

## 9 · Hasil untuk dibandingkan dengan S2

Pisahkan tabel OOF development dan test. Perbandingan dengan S2 E21–E32 memakai **test lawan test pada jawaban yang sama**. Periksa ketiga teks, label, keanggotaan test dan fold; kesamaan ID saja tidak cukup.

Catat pengecualian: E22 menggunakan format konteks berbeda; E29/E31 memakai LR MLP S2 yang berbeda. S2 dengan prediksi identik merupakan kontrol berulang, bukan replikasi independen. Selisih terhadap E22 belum mengisolasi pengaruh mengganti MLP menjadi Ridge.

Unggah ZIP hasil baru untuk audit berpasangan dengan hasil lama. Notebook ini tidak mengarang angka pembanding dari ringkasan ingatan.

In [ ]:
%%aes
show(pd.DataFrame([r for r in result['evaluation'] if r['partition']=='DEVELOPMENT_POOLED_NESTED_OOF']), 'Development: nested pooled OOF')
show(pd.DataFrame([r for r in result['evaluation'] if r['partition']=='TEST_HISTORICAL_HOLDOUT']), 'Test: final Ridge tiga input')
show(pd.DataFrame(result['evaluation']), 'Evaluasi lengkap; train optimistis dan baseline diberi label terpisah')
show(pd.DataFrame(result['alpha_search']), 'Seleksi alpha; bukan evaluasi test')
show(pd.DataFrame([r for r in result['predictions'] if r['partition']=='TEST_HISTORICAL_HOLDOUT']), 'Prediksi test per jawaban; tidak dibulatkan untuk menghitung error')

## 10 · Lima Excel, model final dan unduhan ZIP

Paket berisi dataset/split, fitur/koefisien, prediksi OOF dan test dalam sheet terpisah, evaluasi/seleksi alpha, pengaturan/referensi. Juga disertakan model NPZ tanpa pickle, split JSON, kontrak alpha sebelum test dan manifest checksum. **File hasil privat.**

Dasar metode: [BGE-M3, Chen et al.2024](https://arxiv.org/abs/2402.03216), [Ridge, Hoerl & Kennard1970](https://doi.org/10.1080/00401706.1970.10488634), [bias seleksi model, Cawley & Talbot2010](https://www.jmlr.org/papers/v11/cawley10a.html).
Paper mendukung komponen metode dan pemisahan evaluasi; tidak menjamin error di bawah5poin atau menetapkan alpha/budget token optimal untuk data ini.

In [ ]:
%%aes
download_info = export_ridge_three_input(bundle, X, manifest, cfg, result)
dump(ROOT/'outputs/bge_ridge_download.json', download_info)
show(pd.DataFrame([download_info]), 'Paket hasil siap diunduh')

In [ ]:
info=json.loads((ROOT/'outputs/bge_ridge_download.json').read_text())
archive=Path(info['archive'])
if not archive.is_file():
    raise FileNotFoundError('Paket belum selesai; periksa cell sebelumnya.')
print('ZIP hasil:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))
else:
    print('Ambil file lokal di path di atas.')